# Tara N1 Pretraining

- **Data set:** 5B token subset of FineWeb
- **model.py:** `https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/model.py`
- **train_utils.py:** `https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/train_utils.py`
- **tara_n1_pretrain.pth:** `https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/Models/tara_n1_pretrain.pth`
## Completed training phases:
- ✅Completed first 100M - resulting 100M token corpus
- ✅Completed next 100M - resulting 200M token corpus
- ✅Completed next 100M - resulting 300M token corpus
- ✅Completed next 100M - resulting 400M token corpus
- ✅Completed next 100M - resulting 500M token corpus
- ✅Completed next 500M - resulting 1B token corpus
- ✅Completed next 500M - resulting 1.5B token corpus
- ✅Completed next 500M - resulting 2B token corpus
- ✅Completed next 500M - resulting 2.5B token corpus
- ✅Completed next 500M - resulting 3B token corpus
- ✅Completed next 1B - resulting 4B token corpus
- ✅Completed next 1B - resulting 5B token corpus

## Retraining:
- Training on 5B token subset of FineWeb-edu

In [2]:
import torch
from torch import nn
import tiktoken
import requests
import os

tokenizer = tiktoken.get_encoding("gpt2")
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [3]:
model_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/model.py")
with open("model.py", "w") as f:
    f.write(model_res.text)
print("Downloaded model.py successfully.")


train_utils_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/train_utils.py")
with open("train_utils.py", "w") as f:
    f.write(train_utils_res.text)
print("Downloaded train_utils.py successfully.")

model_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/Models/tara_n1_pretrain.pth")
with open("tara_n1_pretrain.pth", "wb") as f:
    f.write(model_res.content)
print("Downloaded wieghts")

Downloaded model.py successfully.
Downloaded train_utils.py successfully.
Downloaded wieghts


In [4]:
from model import *
from train_utils import *

In [5]:
config = GPTConfig(
    vocab_size=tokenizer.n_vocab,
    block_size=1024,
    batch_size=8,
    d_model=512,
    hidden_layers=2048,
    n_heads=8,
    n_layers=10,
)

# The Dataset

In [6]:
from datasets import load_dataset
from tqdm.auto import tqdm
import numpy as np
# target_tokens = 100_000_000
# skip_tokens = 0

# skip_tokens = 100_000_000
# target_tokens = 200_000_000
# fname = "tokens_100m_to_200m.bin" 

# skip_tokens = 200_000_000
# target_tokens = 300_000_000
# fname = "tokens_200m_to_300m.bin"

# skip_tokens = 4_000_000_000
skip_tokens = 0
train_ds = load_dataset("HuggingFaceFW/fineweb-edu", split="train", name="sample-10BT", streaming=True)
test_ds = load_dataset("HuggingFaceFW/fineweb-edu", split="train", name="sample-10BT", streaming=True)
# train_ds = train_ds.shard(num_shards=10, index=4)
# test_ds = test_ds.shard(num_shards=10, index=4)

train_dataset = StreamingDataset(train_ds, tokenizer, block_size=config.block_size, tokenize_batch_size=64, skip_tokens = skip_tokens)
test_dataset = StreamingDataset(test_ds, tokenizer, block_size=config.block_size, tokenize_batch_size=64, skip_tokens = 5_000_000_000)


train_dataloader = DataLoader(train_dataset, batch_size=config.batch_size, pin_memory=True, num_workers = 2)
test_dataloader = DataLoader(test_dataset, batch_size=config.batch_size, pin_memory=True, num_workers = 0)


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

# Pretraining the model

In [7]:
modelV2 = CustomGPT(config)
state_dict = torch.load("tara_n1_pretrain.pth", map_location=device)
state_dict = {k.removeprefix("module."):v for k, v in state_dict.items()}

# modelV2.load_weights("tara_n1_pretrain.pth")
# modelV2.load_weights("Models/tara_n1_pretrain.pth")

if torch.cuda.device_count() > 1:
    modelV2 = nn.DataParallel(modelV2)
    print(f"Using {torch.cuda.device_count()} GPUs")

modelV2.to(device)

calc_params(modelV2)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(modelV2.parameters(), lr=1e-4)
scaler = torch.amp.GradScaler()

Using 2 GPUs
Total Parameters: 83,562,577
Trainable Parameters: 83,562,577


In [8]:
from tqdm.auto import tqdm
steps = 76294
testing_step = 10000
train_iter = iter(train_dataloader)
avg_train_loss = 0
for step in tqdm(range(1, steps+1)):
    # modelV1.train()
    modelV2.train()
    # def train_step(model, train_dataloader, train_iter, loss_fn, optimizer, scaler, device):
    # train_loss, train_iter = train_step(modelV1, train_dataloader, train_iter, loss_fn, optimizer, scaler, device)
    train_loss = train_step(modelV2, train_iter, loss_fn, optimizer, scaler, device)
    avg_train_loss += train_loss
    
    if step % testing_step == 0:
        # modelV1.eval()
        avg_train_loss /= testing_step
        modelV2.eval()
        
        # def test_step(model, test_dl, n_steps, loss_fn, device):
        # test_loss = test_step(modelV1, test_dataloader, 20, loss_fn, device)
        test_loss = test_step(modelV2, test_dataloader, 20, loss_fn, device)
        print(f"Step {step} | Train Loss: {avg_train_loss} | Test Loss: {test_loss:}")
        avg_train_loss = 0

  0%|          | 0/76293 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# torch.save(modelV1.state_dict(), "tara_n1_pretrain_v1.pth")
# torch.save(modelV2.state_dict(), "tara_n1_pretrain_v2.pth")
# torch.save(modelV2.state_dict(), "tara_n1_pretrain_v3.pth")
# torch.save(modelV2.state_dict(), "tara_n1_pretrain_v4.pth")
# torch.save(modelV2.state_dict(), "tara_n1_pretrain_v5.pth")
# torch.save(modelV2.state_dict(), "tara_n1_pretrain_v6.pth")
# torch.save(modelV2.state_dict(), "tara_n1_pretrain_v7.pth")
# torch.save(modelV2.state_dict(), "tara_n1_pretrain_v8.pth")
# torch.save(modelV2.state_dict(), "tara_n1_pretrain_v9.pth")
# torch.save(modelV2.state_dict(), "tara_n1_pretrain_v10.pth")
# torch.save(modelV2.state_dict(), "tara_n1_pretrain_v11.pth")
torch.save(modelV2.state_dict(), "tara_n1_pretrain_v12.pth")

# Testing



In [ ]:
query = "Once upon a time, "

context = torch.tensor(tokenizer.encode(query), dtype=torch.long).unsqueeze(0).to(device)

# modelV1.eval()
test_model = CustomGPT(config)
test_model.load_weights("tara_n1_pretrain_v12.pth")
# test_model.load_weights("tara_n1_pretrain.pth")
test_model.to(device)
test_model.eval()
with torch.inference_mode():
    # output = modelV1.generate(context, max_new_tokens=100)
    output = test_model.generate(context)

print(f"Input:\n{query}\n")
print(f"Output:\n{tokenizer.decode(output[0].tolist())}")
